# MPPP — 02 Train the mask model (optional)

Not needed to process images: MPPP ships a trained model (`mppp_mask_v2`, downloaded on first use). This notebook retrains it from a labelled mask set and audits the labels.

1. **Regenerate** `images/` from the PDS products for every exported mask (MPPP 8-bit, linear, padded to the detector frame — exactly what the network sees at inference).
2. **Train** ConvNeXt-tiny + FPN/ASPP with a stride-4 decoder (defaults: 6 epochs, lr 5e-5, weight decay 5e-3, horizontal flips, bf16 where supported). ImageNet-22k backbone weights: `checkpoints/convnext_tiny.fb_in22k.safetensors` if present, else Hugging Face.
3. **Check** on held-out frames, and **audit** every training frame against its mask (worst frames by % of the frame wrong) to find bad labels and images.
4. **Promote** the new model to `checkpoints/convnext_tiny_s4_seg_best.pt`; to release it, export it to safetensors (docs/RELEASING.md).

Checkpoints go to `mppp.paths.checkpoints_dir()`: `$MPPP_CHECKPOINTS`, else `checkpoints/` in the repository.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda duplicate OpenMP guard; before torch
import sys
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import mppp
print("mppp", mppp.__version__, "from", Path(mppp.__file__).parent)
import torch
from mppp.paths import checkpoints_dir
from mppp.mask.train import TrainConfig, scan_dataset, grouped_split, train, resolve_precision, promote_checkpoint
CKPT_DIR = checkpoints_dir(create=True)                   # or Path("D:/code/MPPP/checkpoints")
print("MPPP", mppp.__version__, "| checkpoints:", CKPT_DIR)
print("CUDA:", torch.cuda.is_available(), "| bf16:", torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
      "| precision auto ->", resolve_precision("auto", "cuda" if torch.cuda.is_available() else "cpu"))

## Step 1 — regenerate `images/` from the PDS products (run once; resumable)

For each `masks/<stem>.png`, find `<stem>.IMG` anywhere under `pds_dir` (the exact version, else the highest version of the same product), run the MPPP pipeline (no mask inference), and write `images/<stem>.png`: 8-bit RGB + alpha = a copy of your mask (for viewing; training reads only `masks/`). The image must have the mask's size or it is not written.

* An `images/` folder not made by this step is renamed `images_prev_<time>`; nothing is deleted.
* Interrupt and rerun at will: images already written are skipped.
* `_mppp_regen_log.csv` in `images/` has one row per mask (ok / version_substituted / missing / size_mismatch / error).
* To rebuild `images_variable/` from the new images afterwards: `mppp.mask.dataset.build_variable_images(data_dir, reuse_from=[...])`.
* About 1–3 s per image per worker; 5120 px frames need ~1.5 GB RAM per worker. If `workers>1` gives trouble in Jupyter on Windows, use `workers=1`. Try `limit=20` first if you like.

In [ ]:
data_dir = Path("D:/masks/masks_training_set_v7")
pds_dir = Path("D:/data/m2020")
RUN_REGEN = True                       # False once images/ is complete

if RUN_REGEN:
    from mppp.mask.dataset import regenerate_images_from_pds
    regen = regenerate_images_from_pds(data_dir, pds_dir, masks_dir="masks", images_dir="images",
                                       alpha="mask", workers=4, limit=None)

In [ ]:
# look at a few regenerated frames: images | images_variable | mask outline
import csv, random, cv2
from IPython.display import Image, display
from mppp.mask.dataset import render_check
log = list(csv.DictReader(open(data_dir / "images" / "_mppp_regen_log.csv", encoding="utf-8")))
bad = [r for r in log if r["status"] not in ("ok",) and not r["status"].startswith("version_substituted")]
print(len(log), "log rows;", len(bad), "not written:", sorted({r["status"] for r in bad}))
for r in bad[:15]: print("  ", r["status"], r["name"], r["note"])
pngs = sorted(p.name for p in (data_dir / "images").glob("*.png"))
names = [n for n in random.Random(1).sample(pngs, min(4, len(pngs))) if (data_dir / "images_variable" / n).is_file()]
cv2.imwrite(str(data_dir / "regen_check.png"), render_check(data_dir, names))
display(Image(filename=str(data_dir / "regen_check.png")))

## Step 2 — train

In [ ]:
# Labels = the masks/ files.  The alpha channel of images/ and images_variable/ is never read
# (it is only a copy written when those images were made, and is stale once masks/ is edited).
items = scan_dataset(data_dir, image_dirs=("images", "images_variable"), mask_dir="masks",
                     missing="skip")   # images_variable may hold frames whose mask was removed
tr, va = grouped_split(items, val_ratio=0.1, seed=42)
print(f"{len(items)} images, {len({i.mask for i in items})} distinct masks -> train {len(tr)}, val {len(va)} frames")

In [ ]:
from datetime import date
cfg = TrainConfig(
    backbone="convnext_tiny", stride4=True, hflip=True,
    epochs=6, batch_size=4, lr=5e-5, weight_decay=5e-3,
    precision="auto", threshold=0.5, input_size=1648, canvas=(1664, 1248), print_every=50,
    val_preview_every=200, n_val_preview=10,
    quad_split_above=None,   # off; 4000 would split only the 5120 px frames into quadrants, 2500 also the 2560 px ones
    quad_overlap=64, skip_empty_quads=True,
    num_workers=2,           # background image loading; 0 = load in this process
    pretrained_file="auto",  # checkpoints/<backbone>.fb_in22k.safetensors if present, else Hugging Face / its cache
    init_from=None,          # or a ConvNeXtSeg checkpoint to start from
)                            # = TrainConfig()

out = CKPT_DIR / f"{cfg.backbone}{'_s4' if cfg.stride4 else ''}_seg_{date.today():%Y%m%d}.pt"
# Progress: <checkpoint>_debug/ (curves.png, val/, batch/, log.csv); live=True opens a viewer window.
ckpt = train(items, out, cfg, live=True)
print("saved", ckpt)

## Look at progress (re-run any time)

In [ ]:
from IPython.display import Image, display
dbg = out.parent / f"{out.stem}_debug"
display(Image(filename=str(dbg / "curves.png")))
last_val = sorted(p for p in (dbg / "val").glob("*.png") if ".tmp." not in p.name)
if last_val: display(Image(filename=str(last_val[-1])))

## Check the new model on held-out frames (whole frames, as in production)

In [ ]:
import cv2
from mppp.mask import infer_mask
from mppp.mask.monitor import render_rows
picks, seen_cam = [], set()
for it in va:                                   # one held-out frame per camera family, from images/ only
    fam = Path(it.image).name[0]
    if Path(it.image).parent.name == "images" and fam not in seen_cam:
        seen_cam.add(fam); picks.append(it)
rows = []
for it in picks[:4]:
    img = cv2.cvtColor(cv2.imread(it.image), cv2.COLOR_BGR2RGB)
    mask, prob, card = infer_mask(img, ckpt, device="auto")
    rows.append((img, cv2.imread(it.mask, cv2.IMREAD_GRAYSCALE), prob, Path(it.image).name))
panel, ious = render_rows(rows, f"{ckpt.name}  held-out frames", card["threshold"])
cv2.imwrite(str(ckpt.with_suffix(".check.png")), panel)
display(Image(filename=str(ckpt.with_suffix(".check.png"))))
print({k: card[k] for k in ("backbone", "stride4", "quad_split_above", "val_iou", "threshold", "canvas")},
      [round(v, 4) for v in ious])

## Worst frames — label and image problems

The model is run on every frame of `images/` (each mask once, on its production image), exactly as in production, and every frame gets these statistics, all in % of the frame:

| statistic | what it finds |
|---|---|
| `error_pct` | missed + false pixels: what the frame adds to the training error (**default**) |
| `interior_error_pct` | errors farther than `band_px` (0.4 % of the long side: 20 px at 5120, 7 px at 1648) from any label edge: missing or wrong regions, not boundary jitter |
| `confident_error_pct` | the model is sure (P > 0.9 or < 0.1) and the label disagrees: the best single pointer to a wrong label |
| `iou` | ascending; frames with < 1 % terrain are left out (their IoU is 0 or 1 by chance) |

* **train** frames score optimistically (the model has seen them): a *train* frame near the top is a strong label suspect.
* Change `SORT_BY` / `SPLIT` and rerun the table and panel cells: re-ranking uses the saved CSV, no new inference. A CSV from before v0p12 lacks the new columns and is recomputed once.
* Files: `<checkpoint>_debug/frame_audit.csv` (all frames), `worst_frames_NN.png`, `worst_frames.txt` (names, e.g. to select in Metashape).
* About 0.1–0.3 s per frame on the GPU. `image_dirs=None` also scores `images_variable/`; `limit=200` for a quick look.

In [ ]:
from mppp.mask.audit import audit_frames, render_worst, summarize, worst_table, read_audit_csv, rank_rows, SORT_KEYS
# ckpt = CKPT_DIR / "convnext_tiny_s4_seg_YYYYMMDD.pt"   # to audit another model (run the items cell first)
audit_csv = ckpt.parent / f"{ckpt.stem}_debug" / "frame_audit.csv"
RERUN_AUDIT = False                     # True recomputes even if the CSV exists (e.g. after cleaning masks)
audit = read_audit_csv(audit_csv) if audit_csv.is_file() and not RERUN_AUDIT else []
scored = [r for r in audit if r.get("status") in (None, "ok")]
if not scored or scored[0].get("interior_error_pct") is None:      # no CSV yet, or one made before v0p12
    audit = audit_frames(items, ckpt, image_dirs=("images",), which="all")
print(summarize(audit))

SORT_BY = "error_pct"      # "interior_error_pct" | "confident_error_pct" | "iou"
SPLIT = None               # "train" | "val" | None = both
view = rank_rows(audit, SORT_BY, split=SPLIT)
print(f"\nranked by {SORT_BY}: {SORT_KEYS[SORT_BY]}")
print(worst_table(view, n=40))

In [ ]:
N_SHOW = 24                             # worst frames to draw, 6 per page
pages = render_worst(view, ckpt, n=N_SHOW, per_page=6, sort_by=SORT_BY)
for p in pages:
    display(Image(filename=str(p)))
print("names:", pages[0].parent / "worst_frames.txt")

## Promote the new model, and release it

`promote_checkpoint` copies the checkpoint (+ card and `.check.png`) to `convnext_tiny_s4_seg_best.pt` in the same folder; the previous best is moved to `archive/`, never deleted. If the current best was trained on the same dataset with a higher val IoU, nothing is replaced unless `force=True`.

To use it for processing: `config["masking"]["checkpoint"] = str(CKPT_DIR / "convnext_tiny_s4_seg_best.pt")`, or install it as the default registry model (`mppp.mask.hub.install_model(...)`). To publish it: docs/RELEASING.md (export to safetensors, upload, update `mppp/data/models.json`).

In [ ]:
PROMOTE = True
if PROMOTE:
    print(promote_checkpoint(ckpt))
# release a new model version (docs/RELEASING.md):
# from mppp.mask.hub import export_safetensors
# export_safetensors(CKPT_DIR / "convnext_tiny_s4_seg_best.pt", name="mppp_mask_v2", update_registry=True)